# 02. Sensor Validation, Data Preprocessing & Feature Engineering

**Project:** Self-Calibrating and Self-Healing Medical Devices (Infusion Pump Target)  
**Phase:** Phase 2 - Foundational Validation & Preprocessing Pipeline  

---

## Overview
This notebook demonstrates:
1. Non-destructive range validation and sensor anomaly detection (freeze, dropout, missing gaps).
2. Documented cleaning of sentinel values in DosePI (`-99.0` and `-1.0`) while preserving raw readings.
3. Zenodo trailing empty column removal and unicode header normalization.
4. Configurable multi-window rolling feature extraction and OLS trend slopes.
5. Physics-informed infusion pump simulation with strict field separation (`sensor`, `reference`, `digital_twin`).


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.utils.config import SensorType, DataProvenance, FaultClass, RAW_DATA_DIR
from src.data.validation import validate_sensor_dataframe, detect_frozen_sensor, detect_sensor_dropout
from src.data.preprocessing import clean_dosepi_dataframe, clean_sensor_defects_dataframe
from src.data.synthetic_generator import generate_infusion_run, SimulatorConfig
from src.features.rolling_features import RollingFeatureExtractor
from src.features.cross_sensor_features import CrossSensorFeatureExtractor

print("Imports ready.")


In [ ]:
# Inspect Physics-Informed Simulator Output
cfg = SimulatorConfig(duration_seconds=600, random_seed=42)
df_sim = generate_infusion_run(scenario=FaultClass.POSITIVE_DRIFT, config=cfg, fault_start_sec=300, drift_rate_mmhg_per_hour=30.0)

print(f"Generated {len(df_sim)} rows.")
print("Provenance:", df_sim['provenance'].iloc[0])
print("Separated fields available:")
for col in df_sim.columns:
    print(f" - {col}")
display(df_sim.head())


In [ ]:
# Feature Engineering Demo
extractor = RollingFeatureExtractor(windows=[15, 60], features=['mean', 'std', 'slope', 'snr'])
roll_feats = extractor.extract_features(df_sim['sensor_pressure_mmhg'], prefix="p_sensor")

cross_ext = CrossSensorFeatureExtractor()
cross_feats = cross_ext.extract(
    df_sim,
    pressure_col="sensor_pressure_mmhg",
    force_col="sensor_force_n",
    flow_col="sensor_flow_ml_h",
    commanded_rate_col="commanded_rate_ml_h",
    digital_twin_col="dt_predicted_pressure_mmhg",
    target_sensor_col="sensor_pressure_mmhg"
)

combined_features = pd.concat([df_sim[['timestamp_sec', 'sensor_pressure_mmhg']], roll_feats, cross_feats], axis=1)
display(combined_features.head())
